# Autenticación en Django Rest Framework

En este ejercicio se implementó autenticación mediante `TokenAuthentication` para proteger el perfil de usuario del eCommerce.

## Configuración de TokenAuthentication

Se agregó `rest_framework.authtoken` a `INSTALLED_APPS` para utilizar autenticación basada en tokens.

In [ ]:
INSTALLED_APPS = [
    # Aplicaciones de Django...
    "rest_framework",
    "rest_framework.authtoken",
    "ejercicios",
]

Después de agregar `rest_framework.authtoken`, se ejecutaron las migraciones necesarias con `python manage.py migrate`.

## Vista protegida del perfil

La vista utiliza `TokenAuthentication` e `IsAuthenticated`. Los datos se obtienen directamente de `request.user`, por lo que cada token permite consultar únicamente el perfil del usuario autenticado.

In [ ]:
from rest_framework.authentication import TokenAuthentication
from rest_framework.permissions import IsAuthenticated
from rest_framework.response import Response
from rest_framework.views import APIView


class UserProfileView(APIView):
    authentication_classes = [TokenAuthentication]
    permission_classes = [IsAuthenticated]

    def get(self, request):
        user = request.user

        return Response(
            {
                "id": user.id,
                "username": user.username,
                "email": user.email,
                "first_name": user.first_name,
                "last_name": user.last_name,
            }
        )

## Rutas de autenticación y perfil

In [ ]:
from django.urls import path
from rest_framework.authtoken.views import obtain_auth_token

from ejercicios.profile_api import UserProfileView


urlpatterns = [
    path(
        "api/token/",
        obtain_auth_token,
        name="api-token",
    ),

    path(
        "api/profile/",
        UserProfileView.as_view(),
        name="api-profile",
    ),
]

# Pruebas realizadas

## Prueba 1: acceso sin autenticación

Se realizó una petición al perfil sin proporcionar token:

GET /api/profile/

Resultado:

HTTP 401 Unauthorized

Respuesta:

Authentication credentials were not provided.

Esto demuestra que un usuario sin autenticar no puede acceder
al perfil.

## Prueba 2: autenticación de prueba1

El usuario `prueba1` realizó login mediante `/api/token/`
y obtuvo correctamente un token de autenticación.

Posteriormente se utilizó el token para consultar:

GET /api/profile/

Resultado:

HTTP 200 OK

Datos obtenidos:

- ID: 3
- Username: prueba1
- Email: prueba1@example.com

La API mostró únicamente el perfil correspondiente a prueba1.

## Prueba 3: autenticación de prueba2

El usuario `prueba2` realizó login y obtuvo un token diferente.

Al consultar:

GET /api/profile/

Resultado:

HTTP 200 OK

Datos obtenidos:

- ID: 4
- Username: prueba2
- Email: prueba2@example.com

Esto demuestra que el mismo endpoint devuelve un perfil
diferente dependiendo del usuario propietario del token.

## Prueba 4: token inválido

Se realizó una petición utilizando un token inválido.

Resultado:

HTTP 401 Unauthorized

Respuesta:

Invalid token.

Esto demuestra que la API rechaza credenciales que no son
válidas.


# Conclusión

Se implementó correctamente autenticación mediante
`TokenAuthentication` en Django REST Framework.

La vista `/api/profile/` está protegida mediante
`IsAuthenticated` y obtiene los datos directamente desde
`request.user`.

Las pruebas con dos usuarios diferentes demostraron que cada
usuario puede consultar únicamente su propio perfil. Además,
las peticiones sin autenticación o con tokens inválidos son
rechazadas con HTTP 401 Unauthorized.
